![image](car.jpeg)

**Car-ing is sharing**, an auto dealership company for car sales and rental, is taking their services to the next level thanks to **Large Language Models (LLMs)**.

As their newly recruited AI and NLP developer, you've been asked to prototype a chatbot app with multiple functionalities that not only assist customers but also provide support to human agents in the company.

The solution should receive textual prompts and use a variety of pre-trained Hugging Face LLMs to respond to a series of tasks, e.g. classifying the sentiment in a car’s text review, answering a customer question, summarizing or translating text, etc.


In [287]:
pip install sacrebleu

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [288]:
# Import necessary packages
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score
from sacrebleu.metrics import BLEU

from transformers import logging, pipeline
logging.set_verbosity(logging.WARNING)

In [289]:
import pandas as pd

# Start your code here!
df = pd.read_csv('./data/car_reviews.csv', sep=';', encoding='utf-8-sig')
df.head()

,Review,Class
0,I am very satisfied with my 2014 Nissan NV SL....,POSITIVE
1,The car is fine. It's a bit loud and not very ...,NEGATIVE
2,"My first foreign car. Love it, I would buy ano...",POSITIVE
3,I've come across numerous reviews praising the...,NEGATIVE
4,I've been dreaming of owning an SUV for quite ...,POSITIVE


In [290]:
print(f"Dataset loaded: {len(df)} reviews\n")

Dataset loaded: 5 reviews



In [291]:
# Load sentiment analysis model
sentiment_classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

Device set to use cpu


In [292]:
# Get predictionsfor all 5 reviews
reviews_list = df['Review'].tolist()
print(f"Classifying {len(reviews_list)} reviews...\n")

Classifying 5 reviews...



In [293]:
# Store model outputs in predicted_labels
predicted_labels = sentiment_classifier(reviews_list)

print("Model outputs stored in 'predicted_labels':")
for i, pred in enumerate(predicted_labels, 1):
    print(f" Review {i}: {pred}")

Model outputs stored in 'predicted_labels':
 Review 1: {'label': 'POSITIVE', 'score': 0.929397702217102}
 Review 2: {'label': 'POSITIVE', 'score': 0.8654273152351379}
 Review 3: {'label': 'POSITIVE', 'score': 0.9994640946388245}
 Review 4: {'label': 'NEGATIVE', 'score': 0.9935314059257507}
 Review 5: {'label': 'POSITIVE', 'score': 0.9986565113067627}


In [294]:
# Extract labels and map to binary {0, 1} - POSITIVE -> 1, NEGATIVE     -> 0
predictions = [1 if pred['label'] == 'POSITIVE' else 0 for pred in predicted_labels]

print(f"\nBinary predictions (stored in 'predictions'): {predictions}")


Binary predictions (stored in 'predictions'): [1, 1, 1, 0, 1]


In [295]:
# Get true labels and map to binary
true_labels = [1 if label == 'POSITIVE' else 0 for label in df['Class'].tolist()]
print(f"True labels: {true_labels}")


True labels: [1, 0, 1, 0, 1]


In [296]:
# Calculate metrics
accuracy_result = accuracy_score(true_labels, predictions)
f1_result = f1_score(true_labels, predictions)

print(f"\n RESULTS:")
print(f" accuracy_result = {accuracy_result:4f}")
print(f" f1_result = {f1_result:4f}")


 RESULTS:
 accuracy_result = 0.800000
 f1_result = 0.857143


In [297]:
# TASK 2: ENGLISH-TO-SPANISH TRANSLATION

# Extract first review
first_review = df['Review'].iloc[0]
print(f'\nFirst review (full):')
print(f"{first_review[:200]}...\n")
# Extrat first two sentences
sentences = first_review.split('.')
first_two_sentences = sentences[0].strip() + '. ' + sentences[1].strip() + '.'
print(f"First two  sentences to translate:")
print(f"{first_two_sentences}\n")

# Load translation model
print("Loading English-to-Spanish translation model...")
translator = pipeline(
    'translation_en_to_es',
    model='Helsinki-NLP/opus-mt-en-es'
)


First review (full):
I am very satisfied with my 2014 Nissan NV SL. I use this van for my business deliveries and personal use. Camping, road trips, etc. We dont have any children so I store most of the seats in my wareho...

First two  sentences to translate:
I am very satisfied with my 2014 Nissan NV SL. I use this van for my business deliveries and personal use.

Loading English-to-Spanish translation model...


Device set to use cpu


In [298]:
# Translate
print("Translating to Spanish...")
translation_output = translator(first_two_sentences, max_length=512)

# Store translated text in translated_review
translated_review = translation_output[0]['translation_text']
print(f"nTranslated text (stored in 'translated_review'):")
print(f"{translated_review}\n")

Translating to Spanish...
nTranslated text (stored in 'translated_review'):
Estoy muy satisfecho con mi Nissan NV SL 2014. Uso esta camioneta para mis entregas de negocios y uso personal.



In [299]:
# Load reference translation
with open('./data/reference_translations.txt', 'r', encoding='utf-8') as f:
    reference_text = f.read().strip()
print(f"Reference translation:")
print(f"{reference_text}\n")

Reference translation:
Estoy muy satisfecho con mi Nissan NV SL 2014. Utilizo esta camioneta para mis entregas comerciales y uso personal.
Estoy muy satisfecho con mi Nissan NV SL 2014. Uso esta furgoneta para mis entregas comerciales y uso personal.



In [300]:
# Calculate BLEU score
from evaluate import load

import evaluate

# Load BLEU metric
bleu = evaluate.load("bleu")

# Compute - returns dictionary
bleu_score = bleu.compute(
    predictions=[translated_review], 
    references=[[reference_text]]
)
print(f" RESULTS:")
print(f" translated_review = '{translated_review}'")
print(f" bleu_score = {bleu_result['bleu']:.4f}")

 RESULTS:
 translated_review = 'Estoy muy satisfecho con mi Nissan NV SL 2014. Uso esta camioneta para mis entregas de negocios y uso personal.'
 bleu_score = 0.3140


In [301]:
# TASK 3: EXTRACTIVE QUESTION ANSWERING

# Get the 2nd review (index 1) - emphasizes brand aspects
second_review = df['Review'].iloc[1]
print(f"\n2nd Review (context):")
print(f"{second_review[:300]}...\n")


2nd Review (context):
The car is fine. It's a bit loud and not very powerful. On one hand, compared to its peers, the interior is well-built. The transmission failed a few years ago, and the dealer replaced it under warranty with no issues. Now, about 60k miles later, the transmission is failing again. It sounds like a t...



In [302]:
# Load QA model - exactly as specified
print("Loading extractive QA model (deepset/minilm-uncased-squad2)...")
qa_model = pipeline(
    'question-answering',
    model='deepset/minilm-uncased-squad2'
)

Some weights of the model checkpoint at deepset/minilm-uncased-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


Loading extractive QA model (deepset/minilm-uncased-squad2)...


In [303]:
# Set up question and context variables as specified
question = 'What did he like about the brand?'
context = second_review

print(f"Question (stored in 'question'): {question}")
print(f"Context (stored in 'context'): 2nd review text\n")

Question (stored in 'question'): What did he like about the brand?
Context (stored in 'context'): 2nd review text



In [304]:
# Get answer
print("Extracting answer...")
qa_result = qa_model(question=question, context=context)

# Store the actual text answer
answer = qa_result['answer']

print(f" RESULTS:")
print(f" question = '{question}'")
print(f" context = <2nd review text>")
print(f" answer = '{answer}'")
print(f" Confidence: {qa_result['score']:.4f}")

Extracting answer...
 RESULTS:
 question = 'What did he like about the brand?'
 context = <2nd review text>
 answer = 'ride quality, reliability'
 Confidence: 0.4774


In [305]:
# TASK 4: TEXT SUMMARIZATION

# Get the last review
last_review = df["Review"].iloc[-1]
print(f"nLast reviw (original):")
print(f"{last_review}\n")
print(f"Original length: {len(last_review.split())} words\n")

nLast reviw (original):
I've been dreaming of owning an SUV for quite a while, but I've been driving cars that were already paid for during an extended period. I ultimately made the decision to transition to a brand-new car, which, of course, involved taking on new payments. However, given that I don't drive extensively, I was inclined to avoid a substantial financial commitment. The Nissan Rogue provides me with the desired SUV experience without burdening me with an exorbitant payment; the financial arrangement is quite reasonable. Handling and styling are great; I have hauled 12 bags of mulch in the back with the seats down and could have held more. I am VERY satisfied overall. I find myself needing to exercise extra caution when making lane changes, particularly owing to the blind spots resulting from the small side windows situated towards the rear of the vehicle. To address this concern, I am actively engaged in making adjustments to my mirrors and consciously reducing the freque

In [306]:
# Load summarization model
print("Loading summarization model...")
summarizer = pipeline('summarization', model='facebook/bart-large-cnn')

Loading summarization model...


Device set to use cpu


In [307]:
# Summarize to approximately 50-55 tokens
print("Generating summary (target: 50-55 tokens)...")
summary_output = summarizer(
    last_review,
    max_length=55,
    min_length=50,
    do_sample=False
)

# Store in summarized_text
summarized_text = summary_output[0]['summary_text']

print(f" RESULTS:")
print(f" summarized_text = '{summarized_text}'")
print(f" Summary length: {len(summarized_text.split())} tokens")

Generating summary (target: 50-55 tokens)...
 RESULTS:
 summarized_text = 'The Nissan Rogue provides me with the desired SUV experience without burdening me with an exorbitant payment. Handling and styling are great; I have hauled 12 bags of mulch in the back with the seats down and could have held more. The engine delivers strong'
 Summary length: 45 tokens


In [308]:
print("\n" + "="*80)
print("FINAL SUMMARY - ALL REQUIRED VARIABLES")
print("="*80)

print("\n📊 TASK 1 - Sentiment Classification:")
print(f"   predicted_labels = {predicted_labels}")
print(f"   predictions = {predictions}")
print(f"   accuracy_result = {accuracy_result:.4f}")
print(f"   f1_result = {f1_result:.4f}")

print("\n🌍 TASK 2 - Translation:")
print(f"   translated_review = '{translated_review}'")
# Fix: If bleu_score is a dict, print it directly; else, format as float
if isinstance(bleu_score, dict):
    print(f"   bleu_score = {bleu_score}")
else:
    print(f"   bleu_score = {bleu_score:.4f}")

print("\n❓ TASK 3 - Question Answering:")
print(f"   question = '{question}'")
print(f"   context = <2nd review text>")
print(f"   answer = '{answer}'")

print("\n📝 TASK 4 - Summarization:")
print(f"   summarized_text = '{summarized_text}'")
print(f"   Length: {len(summarized_text.split())} tokens")


FINAL SUMMARY - ALL REQUIRED VARIABLES

📊 TASK 1 - Sentiment Classification:
   predicted_labels = [{'label': 'POSITIVE', 'score': 0.929397702217102}, {'label': 'POSITIVE', 'score': 0.8654273152351379}, {'label': 'POSITIVE', 'score': 0.9994640946388245}, {'label': 'NEGATIVE', 'score': 0.9935314059257507}, {'label': 'POSITIVE', 'score': 0.9986565113067627}]
   predictions = [1, 1, 1, 0, 1]
   accuracy_result = 0.8000
   f1_result = 0.8571

🌍 TASK 2 - Translation:
   translated_review = 'Estoy muy satisfecho con mi Nissan NV SL 2014. Uso esta camioneta para mis entregas de negocios y uso personal.'
   bleu_score = {'bleu': 0.3140322081976493, 'precisions': [0.9090909090909091, 0.8571428571428571, 0.75, 0.631578947368421], 'brevity_penalty': 0.40289032152913307, 'length_ratio': 0.5238095238095238, 'translation_length': 22, 'reference_length': 42}

❓ TASK 3 - Question Answering:
   question = 'What did he like about the brand?'
   context = <2nd review text>
   answer = 'ride quality, rel

In [309]:
import json

results = {
    'task_1_sentiment_classification': {
        'predicted_labels': [str(p) for p in predicted_labels],
        'predictions': predictions,
        'accuracy_result': float(accuracy_result),
        'f1_result': float(f1_result)
    },
    'task_2_translation': {
        'translated_review': translated_review,
        'bleu_score': bleu_score
    },
    'task_3_question_answering': {
        'question': question,
        'context': context,
        'answer': answer
    },
    'task_4_summarization': {
        'summarized_text': summarized_text,
        'token_count': len(summarized_text.split())
    }
}

with open('cto_tasks_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("\nResults saved to: cto_tasks_results.json")


Results saved to: cto_tasks_results.json
